# Entrenamiento YOLO - Conteo de Personas

Este notebook entrena el detector YOLO construido en las clases del proyecto:

- `Data/dataset.py` -> `PeopleDataset`: carga imagenes y anotaciones (Roboflow `_annotations.csv`)
- `models/YOLO.py` -> `YOLO`: backbone + neck + head
- `losses/yolo_loss.py` -> `YOLOLoss`: perdida combinada (bbox CIoU + objectness + clasificacion)
- `callbacks/callbacks.py` -> `Callbacks`: early stopping, checkpoints, reduccion de LR, tensorboard
- `util/` -> decode, NMS y metricas para evaluar

In [ ]:
from Data.transform import Transforms
from util.BoundingBox import BoundingBox
from util.Decoder import Decoder
from util.Metrics import DetectionMetrics
from util.NMS import NonMaximumSuppression
from Data.dataset import PeopleDataset
from models.YOLO import YOLO
from losses.yolo_loss import YOLOLoss
from callbacks.callbacks import Callbacks
from Config import CONFIG
import tensorflow as tf
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
path = "Dataset/"
DST = PeopleDataset(CONFIG)
train, valid, test = DST.load(path)

## Preparacion de los targets

El dataset entrega cajas en formato YOLO `[cls, cx, cy, w, h]` (normalizado).
Para entrenar hay que convertirlas a un tensor denso `(grid, grid, 6)`
con la forma `[tx, ty, tw, th, obj, cls]` que espera `YOLOLoss`.

Cada caja se asigna a la celda de la cuadricula donde cae su centro:

- `tx, ty` = desplazamiento del centro dentro de la celda (fraccion de celda)
- `tw, th` = ancho/alto en pixeles
- `obj` = 1 si hay objeto en la celda, 0 si no
- `cls` = one-hot de la clase (1 clase: person)

In [ ]:
train = DST.prepare_for_training(train)
valid = DST.prepare_for_training(valid)
test = DST.prepare_for_training(test)

image, targets = next(iter(train))
print("Imagen :", image.shape)
print("Targets:", targets.shape)

## Modelo

In [ ]:
model = YOLO(num_classes=CONFIG["NUM_CLASSES"])
model.build((None, CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"], 3))
model.summary()

In [ ]:
loss = YOLOLoss()

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=CONFIG["LEARNING_RATE"]
    ),
    loss=loss
)

In [ ]:
callbacks = Callbacks(
    save_dir=CONFIG["CHECKPOINT_DIR"]
).get_callbacks()

## Entrenamiento

In [ ]:
history = model.fit(
    train,
    validation_data=valid,
    epochs=CONFIG["EPOCHS"],
    callbacks=callbacks,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="valid")
plt.title("Perdida durante el entrenamiento")
plt.xlabel("Epoca")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## Evaluacion

Decodifica las predicciones del modelo en cajas, aplica NMS y compara
contra las anotaciones reales usando Precision, Recall y F1 (IoU >= 0.5).

Se evalua sobre una muestra del split `test` (decodificar las 80x80 celdas es lento).

In [ ]:
decoder = Decoder(
    image_size=CONFIG["IMAGE_SIZE"],
    confidence_threshold=CONFIG["CONFIDENCE_THRESHOLD"]
)
nms = NonMaximumSuppression(
    iou_threshold=CONFIG["NMS_IOU_THRESHOLD"]
)
metrics = DetectionMetrics(
    iou_threshold=CONFIG["NMS_IOU_THRESHOLD"]
)

def targets_to_boxes(targets, image_size=CONFIG["IMAGE_SIZE"]):
    """Convierte targets (grid, grid, 6) en BoundingBox en pixeles."""
    grid_h, grid_w = targets.shape[:2]
    stride_x = image_size / grid_w
    stride_y = image_size / grid_h
    boxes = []
    for row in range(grid_h):
        for col in range(grid_w):
            cell = targets[row, col]
            if cell[4] != 1.0:
                continue
            cx = (col + cell[0]) * stride_x
            cy = (row + cell[1]) * stride_y
            w = cell[2]
            h = cell[3]
            boxes.append(BoundingBox(
                xmin=cx - w / 2,
                ymin=cy - h / 2,
                xmax=cx + w / 2,
                ymax=cy + h / 2,
                confidence=1.0
            ))
    return boxes

In [ ]:
num_batches = 2

totals = {"tp": 0, "fp": 0, "fn": 0}

for image_batch, target_batch in test.take(num_batches):

    predictions = model(image_batch, training=False)

    for i in range(predictions.shape[0]):

        detections = nms.apply(decoder.decode(predictions[i]))
        ground_truth = targets_to_boxes(target_batch[i].numpy())

        result = metrics.evaluate(detections, ground_truth)

        totals["tp"] += result["tp"]
        totals["fp"] += result["fp"]
        totals["fn"] += result["fn"]

report = DetectionMetrics()

final = {
    "tp": totals["tp"],
    "fp": totals["fp"],
    "fn": totals["fn"],
    "precision": report.precision(totals["tp"], totals["fp"]),
    "recall": report.recall(totals["tp"], totals["fn"])
}
final["f1_score"] = report.f1_score(final["precision"], final["recall"])

report.print_metrics(final)

## Visualizacion de predicciones

In [ ]:
from Data.visualization import Visualizer

viz = Visualizer()

image_batch, target_batch = next(iter(test))
predictions = model(image_batch, training=False)

for i in range(min(3, image_batch.shape[0])):

    detections = nms.apply(decoder.decode(predictions[i]))

    pred_yolo = [
        np.asarray([
            0.0,
            (b.center_x) / CONFIG["IMAGE_SIZE"],
            (b.center_y) / CONFIG["IMAGE_SIZE"],
            b.width / CONFIG["IMAGE_SIZE"],
            b.height / CONFIG["IMAGE_SIZE"],
        ], dtype=np.float32)
        for b in detections
    ]

    viz.show(image_batch[i], pred_yolo)